In [1]:
import os
# Прячем вторую видеокарту, чтобы Unsloth не ругался
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Очищаем кэш и ставим правильные, проверенные версии библиотек
!pip cache purge
!pip install --no-deps xformers --index-url https://download.pytorch.org/whl/cu121
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl==0.24.0" "peft" "accelerate" "bitsandbytes"

Files removed: 0
Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 96.8 MB/s eta 0:00:00:00:0100:01
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kgbzpv32/unsloth_40e19d89e64745708b99e7321100d835
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kgbzpv32/unsloth_40e19d89e64745708b99e7321100d835
  Resolved https://github.com/unslothai/unsloth.git to commit 17029818f1859a214b4cf9e1a16eae85fb9088f2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 95.7 

In [2]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset

# ==========================================
# 1. ЗАГРУЖАЕМ "ЧЕРНОРАБОЧЕГО" (0.5B параметров)
# ==========================================
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct", # <-- В 3 раза меньше предыдущей!
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Надеваем на него "Ошейник послушания" (LoRA-адаптер)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth", random_state = 3407,
)

# ==========================================
# 2. ОБУЧАЮЩИЙ ДАТАСЕТ (Хаос -> JSON)
# ==========================================
# Мы показываем модели, как извлекать сущности. Обрати внимание на формат output!
data = [
    {
        "input": "Короче, завтра в 15:00 созвон с Максом по дизайну, а в пятницу нужно сдать отчет.", 
        "output": '{\n  "tasks": [\n    {"action": "Созвон по дизайну", "person": "Макс", "deadline": "завтра 15:00"},\n    {"action": "Сдать отчет", "person": null, "deadline": "в пятницу"}\n  ]\n}'
    },
    {
        "input": "Купить молоко, хлеб и попросить Аню скинуть доступы от базы вечером.", 
        "output": '{\n  "tasks": [\n    {"action": "Купить молоко, хлеб", "person": null, "deadline": null},\n    {"action": "Скинуть доступы от базы", "person": "Аня", "deadline": "вечером"}\n  ]\n}'
    },
    {
        "input": "Напомни скинуть доки в бухгалтерию.", 
        "output": '{\n  "tasks": [\n    {"action": "Скинуть доки в бухгалтерию", "person": null, "deadline": null}\n  ]\n}'
    },
    {
        "input": "В понедельник с утра планерка с инвесторами, потом обед с Сашей.", 
        "output": '{\n  "tasks": [\n    {"action": "Планерка с инвесторами", "person": null, "deadline": "в понедельник с утра"},\n    {"action": "Обед", "person": "Саша", "deadline": "в понедельник днем"}\n  ]\n}'
    }
]
# Искусственно размножаем датасет для тренировки
data = data * 50 
dataset = Dataset.from_list(data)

# Строгий промпт, не оставляющий свободы для творчества
underpeople_prompt = """Ты — системный процесс для извлечения данных (Data Extractor).
Твоя задача: прочитать неструктурированный текст и вернуть список задач СТРОГО в формате валидного JSON.
Запрещено писать любые другие слова, приветствия или комментарии. Только JSON.

Текст пользователя:
{}

JSON-результат:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = []
    for inp, out in zip(examples["input"], examples["output"]):
        # Подставляем текст и ожидаемый JSON
        text = underpeople_prompt.format(inp, out) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

mapped_dataset = dataset.map(formatting_prompts_func, batched = True,)
print("\nМодель и данные успешно загружены!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.3.4 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]


Модель и данные успешно загружены!


In [3]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Запускаем процесс "Дрессировки Подчеловека"
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer, 
    train_dataset = mapped_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 60 шагов нам хватит, чтобы вбить ему в голову формат JSON
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
10,1.807789
20,0.361983
30,0.050917
40,0.013202
50,0.012030
60,0.010791


In [4]:
# 1. Переводим "Подчеловека" в рабочий режим (Inference)
FastLanguageModel.for_inference(model)

# 2. Генерируем реальный, спутанный хаос (как голосовое из Telegram)
messy_text = "Слушай, забыл сказать. Короче, в четверг у нас жесткий дедлайн по презентации для Apple, надо еще пнуть Игоря насчет макетов. Ну и да, не забудь заказать пиццу на вечер пятницы, а то голодные останемся."

# Формируем запрос
inputs = tokenizer(
[
    underpeople_prompt.format(
        messy_text, 
        "" # Оставляем пустым, чтобы он сам заполнил JSON
    )
], return_tensors = "pt").to("cuda")

# 3. Заставляем его работать
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

print("\n=== ВХОДЯЩИЙ ХАОС (Голосовое сообщение) ===")
print(messy_text)
print("\n=== ОТВЕТ 'ПОДЧЕЛОВЕКА' (Чистый JSON) ===")
print(response.split("JSON-результат:\n")[1])

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 


=== ВХОДЯЩИЙ ХАОС (Голосовое сообщение) ===
Слушай, забыл сказать. Короче, в четверг у нас жесткий дедлайн по презентации для Apple, надо еще пнуть Игоря насчет макетов. Ну и да, не забудь заказать пиццу на вечер пятницы, а то голодные останемся.

=== ОТВЕТ 'ПОДЧЕЛОВЕКА' (Чистый JSON) ===
{
  "tasks": [
    {"action": "Слушай, забыл сказать", "person": null, "deadline": null},
    {"action": "Веди перескок с Игорем", "person": "Иван", "deadline": "вечер пятни"}
  ]
}


In [5]:
# Сохраняем адаптер (он будет весить вообще копейки, мегабайт 15-20)
model.save_pretrained("underpeople_lora")
tokenizer.save_pretrained("underpeople_lora")

('underpeople_lora/tokenizer_config.json',
 'underpeople_lora/chat_template.jinja',
 'underpeople_lora/tokenizer.json')